In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [4]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [5]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [6]:
from minsearch import Index

index = Index(
    text_fields={'content':1.0},
    keyword_fields=['filename']
)
index.fit(documents)

In [7]:
question="How does the agentic loop keep calling the model until it stops?"

In [8]:
search_results = index.search(
    question,
    boost_dict={'content':1.0},
  
    num_results=5
)



In [9]:
search_results[0]

{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry 

## Homework Questions

In [10]:
len(documents)

72

### 1. How many lesson pages (1 point)

- 24  
- **72**  
- 240  
- 720  

### 2. Indexing and searching (1 point)

- 01-agentic-rag/lessons/03-rag.md  
- **01-agentic-rag/lessons/14-agentic-loop.md**  
- 04-evaluation/lessons/13-llm-as-judge.md  
- 06-best-practices/lessons/02-hybrid-search.md  

Now we will build a RAG assistant on top of this data. Let's use the rag helper script we prepared during the lessons:

wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
RAGBase was written for the FAQ schema (section/question/answer), while our documents have filename and content.

Two solutions are possible:

Implement the RAG flow yourself
Take RAGBase and change the parts related to the FAQ schema - search (to use our index) and build_context
Build a RAG over the index from Q2 and answer the query:

How does the agentic loop keep calling the model until it stops?

Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for this request?


We count input tokens instead of price because the cost depends on the model and provider you use, but the size of the prompt we send is the same for everyone.

Most LLM APIs report token usage on the response object (e.g. response.usage.input_tokens / prompt_tokens). We'll read the input tokens from there.

You will need to modify the code for the rag helper to expose the usage.

In the RAG Helper class, llm returns only the text. Modify it to return the whole response, and change rag to return both the answer and usage (as a tuple or create a small dataclass for that).

In [11]:
from rag_helper import RAGBase
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [12]:
assistant = RAGBase(index, openai_client)
assistant.rag(question)

('The agentic loop keeps calling the model in a `while True` loop. After each model response, it checks whether there were any `function_call` items. If there were, it runs the tools, appends the tool outputs to `messages`, and calls the model again.\n\nIt stops when the model returns a response with **no function calls**:\n\n```python\nif has_function_calls == False:\n    break\n```\n\nSo the exit condition is simply: **no tool calls this turn means the agent is done**.',
 7033,
 111)

### 3. RAG (1 point)

- 700  
- **7000**  
- 70000  
- 700000  

In [13]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [14]:
len(chunks)

295

In [15]:
chunks[1]

{'start': 1000,
 'content': 'the next\nword based on what you typed so far.\n\nA large language model does the same thing, but at a much larger scale.\nIt has billions of parameters and is trained on most of the text on the\ninternet. When it predicts the next word, it feels like you\'re talking\nto an intelligent being. It understands what you ask and gives\nmeaningful answers.\n\nIn this course, we treat LLMs as black boxes. We won\'t look inside or\ncover the theory, and we won\'t host a model ourselves. We use an LLM\nprovider and call it over an API. For us, an LLM is a box: text goes in,\ntext comes out.\n\nBut LLMs have limitations:\n\n- Knowledge cutoff: they only know what was in their training data.\n  If you ask about something that happened after training, they won\'t\n  know - or worse, they\'ll make something up.\n- No access to your data: they can\'t see your documents, databases,\n  or internal systems unless you provide that information.\n- Hallucinations: they sometim

### 4. Chunking (1 point)

- 70  
- **295**  
- 1100  
- 4500  

In [16]:
index.fit(chunks)

In [17]:
assistant = RAGBase(index, openai_client)
assistant.rag(question)

('The loop keeps calling the model with a `while True` loop and checks each response for `function_call` items.\n\n- If the model returns any `function_call`, the code runs the tool, appends the tool result to `messages`, and keeps looping.\n- If the model returns only a final `message` and no function calls, `has_function_calls` stays `False`, and the loop breaks.\n\nSo the stop condition is: **no function calls in the current turn**.',
 2218,
 103)

### 5. RAG with chunking (1 point)

- about the same  
- **3x fewer**  
- 10x fewer  
- 30x fewer  

In [1]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [21]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict = {'content':1.0}
    )

In [22]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [23]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [24]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'Search query text to look up in the course FAQ.'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [25]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

### 6. Turning it into an agent (1 point)

- 0  
- 4  
- **10**  
- 20  